In [1]:
from z_matching import *
import numpy as np
from astropy.table import Table
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from wlclusters import *
import os
os.chdir(r"/local/home/ib286534/Documents/Catalogues")

In [2]:
tr1 = Table.read("unified_clusters_gluematch_direct_DETonly_FILTERED_0.6Mpc_20260224_1624462026-02-24T16_24_56.fits")
tr1 = tr1[tr1["DET_CODE_NB"] == 2]

In [3]:
wl_cat = Table.read('all_euclid_detections_noise.fits', format='fits', memmap=True)
wl_cat = wl_cat[wl_cat["SNR"] > 4.5]

In [4]:
wl_cols = ['RA', 'Dec', 'SNR']
z_cols = ['RIGHT_ASCENSION_CLUSTER', 'DECLINATION_CLUSTER', "SNR_CLUSTER", 'Z_CLUSTER']
wl_cat_matched = matchZ(wl_cat, tr1, healpix2rad(2048, 2.3), wl_cols, z_cols)

In [ ]:
bin_edges = np.logspace(np.log10(0.3/cosmo.h), np.log10(3.0/cosmo.h), 9)

In [ ]:
source_cat = Table.read('cosmohub_RR2_lensmc_v1.4_21987_noNaN.fits', format='fits', memmap=True)
source_cat.rename_columns(['SHE_RA', 'SHE_DEC', 'SHE_E1_CORRECTED', 'SHE_E2_CORRECTED', 'SHE_WEIGHT', 'PHZ_MEDIAN'],
       ['RA', 'Dec', 'e_1', 'e_2', 'weight', 'z_p'])

In [ ]:
# Extract shear profiles
shear_profiles = shear_extraction(cluster_cat=wl_cat_matched, 
                                  sources=source_cat, 
                                  bin_edges=bin_edges,
                                  dz=0.1,
                                  cosmo = cosmo)

In [ ]:
all_chains, results = run(cluster_cat=wl_cat_matched, 
                 shear_profiles=shear_profiles,
                 parnames=['log10cdelt', 'log10mdelt'],
                 cosmo=cosmo,
                 delta=200.)

In [ ]:
wl_cat_matched.add_column(10**results['log10mdelt'], name='M200')